In [1]:
!pwd

/trinity/home/a.anokhin


In [38]:
from torch.utils.data import Dataset
import numpy as np
import json
from tqdm.auto import tqdm

class LocalSetMusique(Dataset):
    def __init__(self, path, tokenizer, length = -1, min_context_len = -1, max_context_len = 1e7, type = "qa", anno_type = "real", seed = 52):
        super().__init__()
        self.length = length
        self.min_context_len = min_context_len
        self.max_context_len = max_context_len
        self.type = type
        self.anno_type = anno_type
        self.tokenizer = tokenizer

        np.random.seed(seed)
        self._load_data(path)

    def name(self):
        return 'musique'

    def _load_data(self, path):
        self.tasks = []

        if self.type not in ["qa", "any"] or self.anno_type not in ["real", "any"]:
            return

        with open(path + '/musique_ans_v1.0_dev.jsonl', 'r') as json_file:
            json_list = list(json_file)
            raw_tasks = [(json.loads(json_str), "dev") for json_str in json_list]

        with open(path + '/musique_ans_v1.0_train.jsonl', 'r') as json_file:
            json_list = list(json_file)
            raw_tasks += [(json.loads(json_str), "train") for json_str in json_list]

        for task, partition in tqdm(raw_tasks, "MuSiQue load"):
            context = ""
            for text in task["paragraphs"]:
                title = text["title"]
                paragraph_text = text["paragraph_text"]
                context += f"TITLE: {title}\nTEXT: {paragraph_text}\n\n"
            context_len = len(self.tokenizer(context)["input_ids"])

            if context_len > self.max_context_len or context_len < self.min_context_len:
                continue
            self.tasks.append(Task(
                    "qa", "real", context_len, context, task["answer"], task["question"], "MuSiQue", partition,
            ))

        self.tasks = np.random.permutation(self.tasks)
        if self.length >= 0:
            self.tasks = self.tasks[:self.length]


    def __len__(self):
        return len(self.tasks)

    def __getitem__(self, idx):
        return self.tasks[idx]


class RetrievalMusique(LocalSetMusique):
    """A default version of Musique Dataset. """

    def __init__(self, path, tokenizer, length=-1, min_context_len=-1, max_context_len=1e7,
                 type="qa", anno_type="real", split='train', seed=52):
        self.split = split #possible values: 'eval', 'train', 'all'
        super().__init__(path, tokenizer, length, min_context_len, max_context_len, type, anno_type, seed)

    def _load_data(self, path):
        self.tasks = []

        if self.type not in ["qa", "any"] or self.anno_type not in ["real", "any"]:
            return

        raw_tasks = []
        if self.split in ['eval', 'all']:
            with open(path + '/musique_ans_v1.0_dev.jsonl', 'r') as json_file:
                json_list = list(json_file)
                raw_tasks.extend( [(json.loads(json_str), "dev") for json_str in json_list] )

        if self.split in ['train', 'all']:
            with open(path + '/musique_ans_v1.0_train.jsonl', 'r') as json_file:
                json_list = list(json_file)
                raw_tasks.extend( [(json.loads(json_str), "train") for json_str in json_list] )

        for sample, partition in tqdm(raw_tasks, "MuSiQue load"):
            context = ""
            for text in sample["paragraphs"]:
                title = text["title"]
                paragraph_text = text["paragraph_text"]
                context += f"TITLE: {title}\nTEXT: {paragraph_text}\n\n"
            context_len = len(self.tokenizer(context)["input_ids"])

            if self.min_context_len <= context_len <= self.max_context_len:
                self.tasks.append(self._adapt_raw_sample(sample))

        self.tasks = np.random.permutation(self.tasks)
        if self.length >= 0:
            self.tasks = self.tasks[:self.length]

    def _adapt_raw_sample(self, sample):
        """Adapt sample to unified format expected by the model"""
        return sample
    
class SimpleEnvAdapter(Dataset):
    """
    Simple adapter that adapts datasets Babilong, HotPotQA and MUSIQUE for QAREtreievalEnv.
    This adapter doesn't tokenize or embeds text chunks.

    You can create different adapter that for example tokenize every text in a sample or
    build faiss index over text chunks.
    """

    def __init__(self, dataset):
        super().__init__()
        self.dataset = dataset
        self.dataset_name = self.dataset.name()

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        sample = self.dataset[index]
        question = sample["question"]
        if question.endswith("?"):
            question = question[:-1]

        sf_idx = []
        chunks_texts = []
        if self.dataset_name == 'hotpotqa':
            sp_title_set = set()
            sample_id = sample['_id']
            for sup in sample['supporting_facts']:
                sp_title_set.add(sup[0])

            for idx, (title, sentences) in enumerate(sample['context']):
                if title in sp_title_set:
                    sf_idx.append(idx)
                chunk = title + " " + " ".join(sentences)
                chunks_texts.append(chunk)

        elif self.dataset_name == 'musique':
            sample_id = sample['id']
            for i, para in enumerate(sample['paragraphs']):
                # if para['is_supporting']:
                #     sf_idx.append(i)
                chunk = para['title'] + '. ' + para['paragraph_text']
                chunks_texts.append(chunk)

            # label order
            for item_json in sample['question_decomposition']:
                sf_idx.append(item_json['paragraph_support_idx'])

        elif self.dataset_name == 'babilong':
            sample_id = index
            for i, sent in enumerate(sample['chunks']):
                chunks_texts.append(sent)

            for i in sample['references_idx']:
                sf_idx.append(i)

        return {
            'id': sample_id,
            'question': question,
            'answer': sample["answer"],
            'chunks_texts': chunks_texts,
            'sf_idx': sf_idx,
        }

In [39]:
from transformers import AutoTokenizer
seed = 42
split = 'eval'
tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')
path = '/trinity/home/a.anokhin/rmt_other_datasets/data/dataloaders/data_sources/musique'

dataset = RetrievalMusique(
    path=path, tokenizer=tokenizer, length=-1,
    min_context_len=0, max_context_len=1e7,
    type='any', anno_type='any', split=split, seed=seed
)

MuSiQue load:   0%|          | 0/2417 [00:00<?, ?it/s]

In [34]:
dataset[0]

{'id': '2hop__121145_561444',
 'paragraphs': [{'idx': 0,
   'title': 'The Real L Word',
   'paragraph_text': 'The Real L Word is an American reality television series aired on the cable station Showtime, where it premiered on June 20, 2010. The show was created by executive producer Ilene Chaiken and Magical Elves Productions, following the success of the television drama "The L Word" also created by Chaiken. "The Real L Word" follows a group of lesbians in their daily lives in Los Angeles, and as of the third season, Brooklyn.',
   'is_supporting': False},
  {'idx': 1,
   'title': 'The Bourne Deception',
   'paragraph_text': 'The Bourne Deception is the title for the novel by Eric Van Lustbader and the seventh novel in the Jason Bourne series created by Robert Ludlum. It was released on June 9, 2009. It is Lustbader\'s fourth Bourne novel, following "The Bourne Sanction," which was published in 2008.',
   'is_supporting': False},
  {'idx': 2,
   'title': 'Taifa of Alpuente',
   'parag

In [40]:
dataset = SimpleEnvAdapter(dataset)

In [22]:
dataset[0]

{'id': '2hop__121145_561444',
 'question': 'Who did the creator of Derech Mitzvosecha follow',
 'answer': 'Dovber Schneuri',
 'chunks_texts': ['The Real L Word. The Real L Word is an American reality television series aired on the cable station Showtime, where it premiered on June 20, 2010. The show was created by executive producer Ilene Chaiken and Magical Elves Productions, following the success of the television drama "The L Word" also created by Chaiken. "The Real L Word" follows a group of lesbians in their daily lives in Los Angeles, and as of the third season, Brooklyn.',
  'The Bourne Deception. The Bourne Deception is the title for the novel by Eric Van Lustbader and the seventh novel in the Jason Bourne series created by Robert Ludlum. It was released on June 9, 2009. It is Lustbader\'s fourth Bourne novel, following "The Bourne Sanction," which was published in 2008.',
  'Taifa of Alpuente. The Taifa of Alpuente was a medieval Berber taifa kingdom that existed from around 1

In [25]:
import sys
import os
sys.path.append(os.path.abspath('/trinity/home/a.anokhin/stage_2/pqn/multi-step-retrieval-rl-pqn-29_05'))
import numpy as np
from collections import namedtuple
from typing import Tuple, Dict, List, Any, Union
import torch.utils
from nltk.probability import gt_demo

# from rl.jax_text_env import TextEnv, TextMemory, TextMemoryItem
from rl.text_env import TextEnv, TextMemory, TextMemoryItem
from transformers import PreTrainedTokenizer, PreTrainedTokenizerFast


class GroundTruthReward:
    def __init__(self, only_at_max_step=False):
        super().__init__()
        self.only_at_max_step = only_at_max_step

    def reward(self, env, action):
        if self.only_at_max_step and (env.num_steps < env.max_steps):
            return 0.

        is_retrieved = []
        for r in env.references:
            is_retrieved.append(r in env.text_state)

        all_retrieved = all(is_retrieved)
        return float(all_retrieved)


class PositionalGTReward(GroundTruthReward):
    """
    This version takes into account position of the support facts.
    In babi tasks several events could have completely identical text descriptions,
    but only one of them can be considered a support fact/reference fact.

    I.E. Merry could visit the same location several times.
    But only the last event allows us to tell where she is at the end of the story.

    This reward takes into account temporal information that allows to distinguish
    true support facts, from similar events.
    """
    def reward(self, env, action):
        if self.only_at_max_step and (env.num_steps < env.max_steps):
            return 0.

        pred_sf = set(map(int, env.memory.item_ids))
        gt_sf = set(env.references_idx)
        return 1.0 if gt_sf.issubset(pred_sf) else 0.0


class BabilongEnv(TextEnv):

    def __init__(self,
                 dataset,
                 max_steps = 3,
                 max_chunks_count = 1000,
                 index_type = "random", # "absolute", "relative"
                 reward_model = GroundTruthReward()):
        
        super().__init__()

        self.dataset = dataset
        self.max_steps = max_steps
        self.max_chunks_count = max_chunks_count
        self.index_type = index_type
        # self.max_embed_length = max_embed_length
        # self.action_embed_length = action_embed_length
        self.reward_model = reward_model

        self.references = None
        self.question = None
        self.sentences = None
        self.facts_idx = None
       
        self.num_steps = 0

    def copy(self):
        return BabilongEnv(self.dataset, 
                           self.max_steps,
                           self.max_chunks_count,
                           self.index_type,
                           self.reward_model)

    def _init_from_sample(self, sample):
        self.references = list(sample['references'])
        self.question = sample['question']  # append as this is a single str
        self.answer = sample['answer']
        self.sentences = np.asarray(sample['chunks'])
        self.facts_idx = list(sample['facts_idx'])
        self.references_idx = sample.get('references_idx', None)
        # self.sentences.extend(sample['noise'])
        # self.sentences.extend(sample['facts'])
        # self.sentences.extend([
        #   f"Fact number {i}: "  + str(f) for i, f in enumerate(sample['facts'])
        # ])
        # self.ref_ids = []
        # for i, f in enumerate(sample['facts']):
        #     if f in self.references:
        #         self.ref_ids.append(i + len(sample['noise']))
        #
        # self.ref_ids = np.array(self.ref_ids)[len(self.ref_ids) - len(self.references):]


    def reset(self, new_sample=None) -> TextMemory:
        if new_sample is not None:
            self._init_from_sample(new_sample)

        elif self.dataset is not None:
            N = len(self.dataset)
            i = np.random.randint(N)
            new_sample = self.dataset[i]
            self._init_from_sample(new_sample)

        self.num_steps = 0

        self.refs_found = []
        self.text_state = []
        
        return super()._reset(self.question, self.sentences)
   

    def step(self, action: int):
        self.num_steps += 1

        done = self.num_steps >= self.max_steps
        
        text_memory, text_item, text_done = super()._step(action)
        self.text_state.append(self.sentences[action])

        r = self._reward(action)
        if r > 1e-5:
            done = True
    
        return text_memory, text_item, r, done or text_done

    @property
    def device(self):
        return self.embedder.device

    def _reward(self, action):
        return self.reward_model.reward(self, action)

    def get_sample_len(self, tokenizer):
        """
        Return total length of all texts in the current retrieval task
        """
        total_len = len(tokenizer(self.question)['input_ids'])
        total_len += sum(len(chunk) for chunk in tokenizer(list(self.sentences))['input_ids'])
        return total_len

In [29]:
class AdaptedBabilongEnv(BabilongEnv):
    def __init__(self,
                 dataset: SimpleEnvAdapter, # Явно указываем тип для ясности
                 max_steps=3,
                 # max_chunks_count больше не нужен здесь, т.к. SimpleEnvAdapter определяет кол-во чанков
                 index_type="random", 
                 reward_model=GroundTruthReward()):
        
        # Вызываем __init__ родителя.
        # BabilongEnv ожидает 'max_chunks_count', но он не будет использоваться для генерации
        # сэмпла, т.к. _init_from_sample переопределен. Поставим 0 или другое значение.
        super().__init__(dataset=dataset, # self.dataset будет SimpleEnvAdapter
                         max_steps=max_steps,
                         max_chunks_count=0, # Не релевантно для AdaptedBabilongEnv
                         index_type=index_type,
                         reward_model=reward_model)

    def _init_from_sample(self, sample):
        # 'sample' здесь приходит от SimpleEnvAdapter
        # Формат sample: {'id', 'question', 'answer', 'chunks_texts', 'sf_idx'}
        
        self.question = sample['question']
        self.answer = sample['answer']
        self.sentences = np.asarray(sample['chunks_texts']) # Это будут наши "чанки"
        
        # 'sf_idx' из SimpleEnvAdapter - это индексы релевантных чанков в 'chunks_texts'
        # Это соответствует 'references_idx' в терминологии BabilongEnv
        self.references_idx = list(sample['sf_idx'])
        
        # 'references' - это тексты релевантных чанков
        self.references = [self.sentences[i] for i in self.references_idx if 0 <= i < len(self.sentences)]
        
        # 'facts_idx' в оригинальном BabilongEnv - это индексы всех "фактов" из задачи bAbI
        # до их смешивания с шумом. SimpleEnvAdapter не предоставляет эту информацию напрямую
        # в таком же виде (он дает 'sf_idx' - индексы *поддерживающих* фактов/параграфов).
        # Для совместимости с логикой награды, которая обычно смотрит на references_idx,
        # мы можем установить facts_idx равным references_idx.
        # Или, если бы была необходимость, можно было бы считать все чанки "фактами":
        # self.facts_idx = list(range(len(self.sentences)))
        # Но для GroundTruthReward, важнее references_idx.
        self.facts_idx = list(sample['sf_idx']) 
        
        # print(f"AdaptedBabilongEnv._init_from_sample: sample keys: {sample.keys()}")
        # print(f"  question: {self.question}")
        # print(f"  sentences len: {len(self.sentences)}")
        # print(f"  references_idx (from sf_idx): {self.references_idx}")
        # print(f"  facts_idx (set to sf_idx): {self.facts_idx}")


    def copy(self):
        # Создаем копию нового класса
        return AdaptedBabilongEnv(dataset=self.dataset, # self.dataset это SimpleEnvAdapter
                                  max_steps=self.max_steps,
                                  index_type=self.index_type,
                                  reward_model=self.reward_model)

    # Методы reset, step, _reward, get_sample_len наследуются от BabilongEnv.
    # reset вызовет наш переопределенный _init_from_sample.
    # step и _reward будут использовать self.references_idx, который мы корректно установили.

In [ ]:
class QARetrievalEnv(TextEnv):  # Наследуемся напрямую от TextEnv
    def __init__(self,
                 dataset: SimpleEnvAdapter, # Явно указываем тип для ясности
                 max_steps=10,
                 index_type="random", # "absolute", "relative" - используется TextEnv?
                 reward_model=GroundTruthReward()):
        
        super().__init__() # Вызов конструктора TextEnv

        self.dataset = dataset # Это будет SimpleEnvAdapter
        self.max_steps = max_steps
        self.index_type = index_type # Если TextEnv его использует, иначе можно убрать
        self.reward_model = reward_model

        # Атрибуты, которые ранее устанавливались в BabilongEnv или его _init_from_sample
        # и используются в методах, которые мы скопируем или уже имеем
        self.references: List[str] = []
        self.question: str = ""
        self.answer: Any = None # Тип ответа может быть разным
        self.sentences: np.ndarray = np.array([]) # Массив текстов "чанков"
        self.facts_idx: List[int] = [] # Индексы "фактов" (в контексте Adapted это sf_idx)
        self.references_idx: List[int] = [] # Индексы релевантных чанков

        self.num_steps: int = 0
        self.text_state: List[str] = [] # Хранит тексты выбранных чанков

        # Атрибуты, которые могли быть в BabilongEnv и используются reward_model или др. логикой
        # self.refs_found = [] # Если используется где-то еще, кроме как локально в BabilongEnv.reset

    def _init_from_sample(self, sample: Dict[str, Any]):
        # 'sample' здесь приходит от SimpleEnvAdapter
        # Формат sample: {'id'(опционально), 'question', 'answer', 'chunks_texts', 'sf_idx'}
        
        self.question = sample['question']
        self.answer = sample['answer']
        self.sentences = np.asarray(sample['chunks_texts']) # Это будут наши "чанки"
        
        # 'sf_idx' из SimpleEnvAdapter - это индексы релевантных чанков в 'chunks_texts'
        # Это соответствует 'references_idx'
        self.references_idx = list(map(int, sample['sf_idx'])) # Убедимся, что это int
        
        # 'references' - это тексты релевантных чанков
        self.references = [self.sentences[i] for i in self.references_idx if 0 <= i < len(self.sentences)]
        
        # 'facts_idx' - для совместимости и если какая-то логика на них полагается.
        # В данном контексте "факты" это и есть поддерживающие чанки.
        self.facts_idx = list(self.references_idx) 
        
        # Опционально: отладочный вывод
        # print(f"QARetrievalEnv._init_from_sample: sample keys: {sample.keys()}")
        # print(f"  question: {self.question}")
        # print(f"  sentences len: {len(self.sentences)}")
        # print(f"  references_idx (from sf_idx): {self.references_idx}")
        # print(f"  facts_idx (set to sf_idx): {self.facts_idx}")

    def reset(self, new_sample: Dict[str, Any] = None) -> TextMemory:
        if new_sample is not None:
            self._init_from_sample(new_sample)
        elif self.dataset is not None:
            # Эта часть предполагает, что self.dataset (SimpleEnvAdapter)
            # поддерживает __len__ и __getitem__ для случайного выбора.
            # Если SimpleEnvAdapter предоставляет другой API (например, .sample()),
            # эту логику нужно будет адаптировать.
            try:
                N = len(self.dataset)
                if N == 0:
                    raise ValueError("Dataset is empty.")
                i = np.random.randint(N)
                sample_from_dataset = self.dataset[i]
                self._init_from_sample(sample_from_dataset)
            except (TypeError, NotImplementedError) as e:
                # Если __len__ или __getitem__ не реализованы, попробуем .sample()
                if hasattr(self.dataset, 'sample') and callable(getattr(self.dataset, 'sample')):
                    try:
                        sample_from_dataset = self.dataset.sample()
                        self._init_from_sample(sample_from_dataset)
                    except Exception as sample_e:
                        raise RuntimeError(f"Failed to get sample using .sample() from dataset: {sample_e}") from sample_e
                else:
                    raise RuntimeError(
                        "Dataset adapter does not support len()/getitem[] for sampling, "
                        "nor does it have a .sample() method. "
                        "Please provide new_sample directly to reset() or adapt the adapter."
                    ) from e
        else:
            # Если нет ни new_sample, ни dataset, это ошибка конфигурации
            raise ValueError("Cannot reset environment: no new_sample provided and no dataset configured.")


        self.num_steps = 0
        self.text_state = [] # Сбрасываем историю выбранных текстов
        self.refs_found = [] # Если этот атрибут был и использовался, его тоже сбрасываем

        # _reset из TextEnv должен инициализировать TextMemory на основе question и sentences
        # TextEnv._reset(self, query: str, chunks: Union[List[str], np.ndarray]) -> TextMemory
        return super()._reset(self.question, self.sentences)
   
    def step(self, action: int): # -> Tuple[TextMemory, TextMemoryItem, float, bool]
                                 # TextMemoryItem определен в rl.text_env
        self.num_steps += 1
        done = self.num_steps >= self.max_steps
        
        # _step из TextEnv должен обновить память и вернуть выбранный элемент
        # TextEnv._step(self, action_idx: int) -> Tuple[TextMemory, TextMemoryItem, bool]
        # где bool это text_done (например, если память переполнилась или все элементы выбраны)
        text_memory, text_item, text_done = super()._step(action)
        
        # Добавляем текст выбранного чанка в self.text_state для GroundTruthReward
        if 0 <= action < len(self.sentences):
            self.text_state.append(self.sentences[action])
        else:
            # Это не должно происходить, если action всегда валидный индекс
            # Можно добавить логгирование или обработку ошибки
            print(f"Warning: Action {action} is out of bounds for sentences (len: {len(self.sentences)})")


        r = self._reward(action)
        # Если достигнута цель (например, все референсы найдены), эпизод может завершиться досрочно
        if r > 1e-5: # Сравниваем с небольшим эпсилон, т.к. награда может быть float
            done = True
    
        return text_memory, text_item, r, done or text_done

    def _reward(self, action: int) -> float:
        # Используем self.reward_model, который был передан в конструкторе
        return self.reward_model.reward(self, action)

    @property
    def device(self):
        # Предполагаем, что TextEnv (родительский класс) управляет эмбеддером
        # и предоставляет доступ к его устройству.
        # Если TextEnv не имеет self.embedder, эту часть нужно адаптировать.
        if hasattr(self, 'embedder') and self.embedder is not None:
            return self.embedder.device
        # Заглушка или ошибка, если embedder не найден (зависит от реализации TextEnv)
        # print("Warning: embedder not found or not initialized in TextEnv for .device property.")
        return "cuda:0" # или None, или raise AttributeError

    def get_sample_len(self, tokenizer) -> int:
        """
        Return total length of all texts (question + chunks) in the current task,
        tokenized by the provided tokenizer.
        """
        if not self.question or self.sentences is None or len(self.sentences) == 0:
            # Это может случиться, если reset() не был вызван или не отработал корректно
            # print("Warning: get_sample_len called before environment was properly reset with data.")
            return 0
        
        total_len = len(tokenizer(self.question)['input_ids'])
        # Убедимся, что sentences - это список строк для токенизатора
        sentences_list = list(self.sentences)
        if sentences_list: # Проверка, что список не пустой
            # Некоторые токенизаторы могут возвращать список списков input_ids, если на вход список строк
            tokenized_chunks = tokenizer(sentences_list)['input_ids']
            total_len += sum(len(chunk_ids) for chunk_ids in tokenized_chunks)
        return total_len

    def copy(self):
        # Создаем копию этого же класса
        # self.dataset (SimpleEnvAdapter) и self.reward_model передаются по ссылке,
        # что обычно является ожидаемым поведением для таких объектов.
        # Если нужна глубокая копия этих объектов, ее нужно реализовать отдельно.
        return AdaptedBabilongEnv(dataset=self.dataset,
                                  max_steps=self.max_steps,
                                  index_type=self.index_type,
                                  reward_model=self.reward_model)

In [136]:
#env = BabilongEnv(dataset)
env = QARetrievalEnv(dataset)

In [163]:
print("\nResetting environment...")
obs_memory = env.reset() # TextMemory object

print(f"Environment reset. Question: '{env.question}'")
print(f"Available sentences (chunks): {len(env.sentences)}")
# print("Sentences:", env.sentences)
print(f"Target reference indices: {env.references_idx}")
print(f"Target reference texts: {env.references}")

done = False
total_reward = 0
for step_num in range(env.max_steps):
    if done:
        break
    action = np.random.randint(0, len(env.sentences)) # Случайное действие
    print(f"\nStep {step_num + 1}, Agent takes action: {action} ('{env.sentences[action][:50]}...')")
    
    obs_memory, text_item, reward, done = env.step(action)
    
    total_reward += reward
    #print(f"Received: text_item='{text_item[:50] if text_item else 'N/A'}...', reward={reward}, done={done}")
    if reward > 0:
        print(f"  Found a reference! Current refs_found: {env.refs_found}")

print(f"\nEpisode finished. Total reward: {total_reward}")


Resetting environment...
Environment reset. Question: 'Who sings home alone tonight with the singer of you can crash my party anytime'
Available sentences (chunks): 20
Target reference indices: [10, 15]
Target reference texts: ['Crash My Party. Crash My Party is the fourth studio album by American country music artist Luke Bryan. It was released on August 13, 2013 by Capitol Nashville. Its first single, the title track, reached number one on the Billboard Country Airplay chart. The album was produced by Jeff Stevens. A deluxe edition with four bonus tracks is available exclusively at Target and Walmart.', "Home Alone Tonight. ``Home Alone Tonight ''is a song recorded by American country music artist Luke Bryan as a duet with Karen Fairchild of American country music group Little Big Town for his fifth studio album, Kill the Lights (2015). Upon the release of the album, the song entered the Billboard Hot Country Songs chart at number 33 on the strength of digital downloads. It was serv